In [51]:
# Algorithm 3 (Guruswami–Sudan with multiplicities) — SageMath, self-contained
# implementation for GRS codes: setup → interpolation (s>1)
# → root-finding (Alekhnovich) → validation + log.



from sage.all import *
from time import perf_counter
from collections import namedtuple
import random

# 1) Setup 

# Field / code parameters
q = 2**8
F = GF(q, name='a')
n, k = 255, 223                      # can switch to (63, 45) for faster demo
wy = k - 1                           # (1, wy) weighted degree for GS (Y-weight = k-1)

# Choose n distinct evaluation points and (optional) column multipliers
alphas = F.list()[:int(n)]           # first n elements of the field
vcol   = [F.one() for _ in range(int(n))]   # all multipliers = 1 (still a valid GRS)

# Build the GRS(n,k) code
C = codes.GeneralizedReedSolomonCode(alphas, int(k), vcol)

# Bind evaluation points / multipliers for later use
alpha = tuple(C.evaluation_points())
vcol  = tuple(C.column_multipliers())

# Polynomial ring for messages
R.<X> = F[]

# encode a polynomial f(X) (deg < k) into a GRS codeword
def encode_poly(f):
    return vector(F, [vcol[i] * f(alpha[i]) for i in range(int(n))])

# Diagnostics: Johnson radius and tau_max(s,ℓ) 

from sage.coding.guruswami_sudan.utils import johnson_radius
from sage.coding.guruswami_sudan.gs_decoder import GRSGuruswamiSudanDecoder

s   = 2                         # multiplicity (Alg-3 needs s>1)
ell = max(s, 20)                # list-size cap on deg_Y Q

J = johnson_radius(int(n), int(n - k + 1))
print(f"Johnson radius τ_J ≈ {float(J):.2f}")

# Optimize ℓ for the chosen s (avoid negative τ when ℓ is too small)
tau_max, (s, ell) = GRSGuruswamiSudanDecoder.guruswami_sudan_decoding_radius(
    n_k=(int(n), int(k)), s=int(s)
)
tau_max = int(max(0, tau_max))   # clamp to non-negative
print(f"Using (s, ℓ)=({s}, {ell}); GS radius τ_max = {tau_max}")

# Pick t so sampling is valid and within radius
t_requested = 20
t = max(0, min(int(t_requested), tau_max))
print(f"Injecting t = {t} errors (requested {t_requested}).")
print(f"Injecting t = {t} errors (requested {t_requested}).")

# Random message
P_true = sum(F.random_element() * X**i for i in range(int(k)-1))
c = encode_poly(P_true)


# Corrupt t positions
rng = random.Random(int(7))
err_positions = rng.sample(range(int(n)), int(t))
y = list(c)
for pos in err_positions:
    y[pos] = F.random_element()
y = vector(F, y)

# 2) Choose (s, ℓ) and degree cap D 
s   = 2
ell = max(s, 20)

def constraints_count(n_, s_):
    return int(n_) * ((s_+1)*s_ // 2)

def monomials_upto_D(D_, wy_, ell_):
    # Monomial set M = {(i,j): i + wy*j <= D, 0 <= j <= ell}
    M_ = []
    for j in range(int(ell_)+1):
        i_max = D_ - wy_*j
        if i_max < 0:
            break
        for i in range(i_max+1):
            M_.append((i, j))
    return M_

# Smallest D so that #variables >= #constraints + 1 (slack)
D = 0
while True:
    M = monomials_upto_D(D, int(wy), ell)
    if len(M) >= constraints_count(n, s) + 1:
        break
    D += 1

num_vars = len(M)
num_cons = constraints_count(n, s)

# 3) Interpolation with multiplicities 

# Work in S = F[X][Y]
S.<Y> = PolynomialRing(R)

mon_to_idx = {mon: idx for idx, mon in enumerate(M)}
A = matrix(F, num_cons, num_vars)

# Interpolate on GRS points: (alpha_i, y_i / v_i)
y_tilde = [y[i] / vcol[i] for i in range(int(n))]

row = 0
tI0 = perf_counter()
for r in range(int(n)):
    xi = alpha[r]
    yi = y_tilde[r]
    # enforce all Hasse partials with a+b < s
    for a in range(s):
        for b in range(s - a):
            for (i_exp, j_exp) in M:
                if a <= i_exp and b <= j_exp:   # avoid negative exponents
                    A[row, mon_to_idx[(i_exp, j_exp)]] = (
                        binomial(i_exp, a) * binomial(j_exp, b)
                        * (xi**(i_exp - a)) * (yi**(j_exp - b))
                    )
            row += 1

ker = A.right_kernel()
if ker.dimension() == 0:
    raise ValueError("No nonzero Q found; increase D or ℓ (or reduce s).")

coef = vector(F, ker.basis()[0])

Q = S.zero()
for (i_exp, j_exp), cij in zip(M, coef):
    if cij != 0:
        Q += cij * (X**i_exp) * (Y**j_exp)

tI1 = perf_counter()
time_interp = tI1 - tI0

# 4) Root-finding (GS) 

# Use Sage's Alekhnovich root finder (from sage impl)
from sage.coding.guruswami_sudan.gs_decoder import alekhnovich_root_finder

tR0 = perf_counter()
# max degree in X for Y-roots is ≤ k-1
candidates_polys = alekhnovich_root_finder(Q, maxd=int(wy))
tR1 = perf_counter()
time_root = tR1 - tR0

# 5) Validation 

def hamming_dist(u, v):
    return sum(1 for a, b in zip(u, v) if a != b)

ranked = []
for P_hat in candidates_polys:
    c_hat = encode_poly(P_hat)                   # use same GRS encoder
    d = hamming_dist(c_hat, y)
    ranked.append((P_hat, d))
ranked.sort(key=lambda t_: t_[1])

# 6) Logging 

from collections import namedtuple

Log = namedtuple("Log", "n k q s ell D num_vars num_constraints time_interp time_root time_total num_candidates best_distance")

time_total = time_interp + time_root

log = Log(
    n=int(n), k=int(k), q=int(q), s=int(s), ell=int(ell), D=int(D),
    num_vars=int(num_vars), num_constraints=int(num_cons),
    time_interp=time_interp, time_root=time_root, time_total=time_total,
    num_candidates=len(candidates_polys),
    best_distance=(ranked[0][1] if ranked else None)
)

print("\n" + "=" * 50)
print("              GS (Algorithm 3) Run Summary              ")
print("=" * 50)

print(f"{'n':<18}: {log.n}")
print(f"{'k':<18}: {log.k}")
print(f"{'q':<18}: {log.q}")
print(f"{'s':<18}: {log.s}")
print(f"{'ℓ (ell)':<18}: {log.ell}")
print(f"{'D':<18}: {log.D}")
print(f"{'# Variables':<18}: {log.num_vars}")
print(f"{'# Constraints':<18}: {log.num_constraints}")
print(f"{'Interp Time (s)':<18}: {log.time_interp:.4f}")
print(f"{'Root Time (s)':<18}: {log.time_root:.4f}")
print(f"{'Total Time (s)':<18}: {log.time_total:.4f}")
print(f"{'# Candidates':<18}: {log.num_candidates}")
print(f"{'Best Distance':<18}: {log.best_distance}")
print("=" * 50)

if ranked:
    print("\nTop-3 Candidates by Distance (deg P ≤ k-1):")
    print("-" * 50)
    for i, (p, d) in enumerate(ranked[:3], start=1):
        print(f"  {i}.  Distance = {d:<8}  Degree(P) = {p.degree()}")
else:
    print("\nNo ranked candidates available.")
print("=" * 50)


Johnson radius τ_J ≈ 17.07
Using (s, ℓ)=(2, 2); GS radius τ_max = 16
Injecting t = 16 errors (requested 20).
Injecting t = 16 errors (requested 20).

              GS (Algorithm 3) Run Summary              
n                 : 255
k                 : 223
q                 : 256
s                 : 2
ℓ (ell)           : 20
D                 : 477
# Variables       : 768
# Constraints     : 765
Interp Time (s)   : 22.0837
Root Time (s)     : 0.5031
Total Time (s)    : 22.5869
# Candidates      : 1
Best Distance     : 16

Top-3 Candidates by Distance (deg P ≤ k-1):
--------------------------------------------------
  1.  Distance = 16        Degree(P) = 221


In [53]:

# Algorithm-3 DEMO (visible) vs Algorithm-2 DEMO
# SageMath kernel — copy-paste-ready (fixed field use)


from sage.all import *
from time import perf_counter
from collections import namedtuple
import random


# 0) Small, low-rate GRS code
q = 2**8
F = GF(q, name='a')
n, k = 63, 21                 # d = 43, unique radius = 21, Johnson ≈ 27.5
wy = k - 1                    # Y-weight for GS (deg_Y cap = wy)

# Evaluation points & multipliers (GRS)
alphas = F.list()[:n]
vcol   = [F.one() for _ in range(n)]
C = codes.GeneralizedReedSolomonCode(alphas, k, vcol)
alpha = tuple(C.evaluation_points())
vcol  = tuple(C.column_multipliers())

R.<X> = F[]

def encode_poly(f):
    """GRS encode a message polynomial f into a length-n codeword over F."""
    return vector(F, [vcol[i] * f(alpha[i]) for i in range(n)])


# 1) Pick a random message and encode (deg < k)
rng = random.Random(int(7))  # deterministic
P_true = sum(F.random_element() * X**i for i in range(k-1))
c_true = encode_poly(P_true)


# 2) Choose t in list-decoding regime (compute τ_max etc.)
from sage.coding.guruswami_sudan.utils import johnson_radius
from sage.coding.guruswami_sudan.gs_decoder import GRSGuruswamiSudanDecoder

J = johnson_radius(n, n - k + 1)
print(f"Johnson radius τ_J ≈ {float(J):.2f}")

# Algorithm-3 settings (fix s, optimize ℓ for that s)
s3 = 2
tau3, (s3, ell3) = GRSGuruswamiSudanDecoder.guruswami_sudan_decoding_radius(n_k=(n, k), s=s3)
tau3 = int(tau3)
print(f"Alg-3 using (s,ℓ)=({s3},{ell3}) ⇒ τ_max={tau3}")

# Inject errors; clamp to τ_max so we see a non-empty list
t_requested = 25
t = min(t_requested, tau3)
print(f"Injecting t = {t} errors (requested {t_requested}).")


# 3) Corrupt and build received word y
err_positions = rng.sample(range(n), t)
y = list(c_true)
for pos in err_positions:
    y[pos] = F.random_element()
y = vector(F, y)

# 4) Visible GS interpolation (+ multiplicities) and root-finding
def gs_visible_decode(alpha, vcol, y, n, k, s, ell, wy):
    """
    Visible GS:
      - Build multiplicity interpolation system for Q(X,Y)
      - Solve linear system over base field
      - Alekhnovich root-finding to extract candidates of deg_Y ≤ wy
    Returns:
      dict(Q, D, M, candidates, time_interp, time_root)
    """
    # Keep the SAME field as input points to avoid coercion issues
    F_local = alpha[0].parent()
    R_loc.<X> = PolynomialRing(F_local)
    S_loc.<Y> = PolynomialRing(R_loc)

    # Monomial set: i + wy*j <= D, 0 <= j <= ell
    def constraints_count(n_, s_):
        return n_ * ((s_ + 1) * s_ // 2)

    def monomials_upto_D(D_):
        Ms = []
        for j in range(ell + 1):
            i_max = D_ - wy * j
            if i_max < 0:
                break
            for i in range(i_max + 1):
                Ms.append((i, j))
        return Ms

    # Minimal D s.t. #monomials > #constraints
    D = 0
    while True:
        M = monomials_upto_D(D)
        if len(M) >= constraints_count(n, s) + 1:
            break
        D += 1

    # Build linear system over F_local
    mon_to_idx = {mon: idx for idx, mon in enumerate(M)}
    A = matrix(F_local, constraints_count(n, s), len(M))

    # Interpolate at (alpha_i, y_i / v_i) for GRS
    y_tilde = [y[i] / vcol[i] for i in range(n)]

    row = 0
    tI0 = perf_counter()
    for r in range(n):
        xi = alpha[r]
        yi = y_tilde[r]
        for a in range(s):
            for b in range(s - a):
                for (i_exp, j_exp) in M:
                    if a <= i_exp and b <= j_exp:
                        A[row, mon_to_idx[(i_exp, j_exp)]] = (
                            binomial(i_exp, a)
                            * binomial(j_exp, b)
                            * (xi ** (i_exp - a))
                            * (yi ** (j_exp - b))
                        )
                row += 1

    ker = A.right_kernel()
    if ker.dimension() == 0:
        return dict(Q=None, D=D, M=M, candidates=[], time_interp=None, time_root=None)

    coef = vector(F_local, ker.basis()[0])

    # Assemble Q(X,Y)
    Q = S_loc.zero()
    for (i_exp, j_exp), cij in zip(M, coef):
        if cij != 0:
            Q += cij * (X ** i_exp) * (Y ** j_exp)

    tI1 = perf_counter()
    time_interp = tI1 - tI0

    # Root-finding (Alekhnovich)
    from sage.coding.guruswami_sudan.gs_decoder import alekhnovich_root_finder
    tR0 = perf_counter()
    candidates = alekhnovich_root_finder(Q, maxd=wy)
    tR1 = perf_counter()
    time_root = tR1 - tR0

    return dict(Q=Q, D=D, M=M, candidates=candidates,
                time_interp=time_interp, time_root=time_root)

# Run Algorithm-3 (s=2, ℓ as optimized above)
res3 = gs_visible_decode(alpha, vcol, y, n, k, s3, ell3, wy)


# 5) Compare with Algorithm-2 (Sudan, s=1) on same y
s2 = 1
tau2, (s2, ell2) = GRSGuruswamiSudanDecoder.guruswami_sudan_decoding_radius(n_k=(n, k), s=s2)
tau2 = int(tau2)
print(f"Alg-2 using (s,ℓ)=({s2},{ell2}) ⇒ τ_max={tau2}")

res2 = gs_visible_decode(alpha, vcol, y, n, k, s2, ell2, wy)

# 6) Validate and show candidates (ranking)
def rank_candidates(cands, P_true):
    """Rank candidates by Hamming distance after GRS re-encoding."""
    def enc_dist(P):
        c_hat = vector(F, [vcol[i] * P(alpha[i]) for i in range(n)])
        return sum(1 for a, b in zip(c_hat, y) if a != b)

    ranked = [(P, enc_dist(P)) for P in cands]
    ranked.sort(key=lambda t: t[1])
    ok = any(P == P_true for P, _ in ranked)
    return ranked, ok

ranked3, has_true3 = rank_candidates(res3['candidates'], P_true)
ranked2, has_true2 = rank_candidates(res2['candidates'], P_true)

# for better print logs
def print_rule(ch="=", n=54):
    print(ch * n)

def print_kv(key, val, width=22, fmt=None):
    if fmt is None:
        print(f"{key:<{width}}: {val}")
    else:
        print(f"{key:<{width}}: {fmt.format(val)}")

def print_algo_block(title, params, res, ranked, has_true, tau):
    print()
    print_rule("=")
    print(f"{title}")
    print_rule("=")

    # Parameters
    print_kv("Parameters", f"s={params['s']}, ℓ={params['ell']}")
    print_kv("GS radius τ_max", tau)
    print_kv("Interpolation degree cap D", res['D'])
    print_kv("# candidates", len(res['candidates']))
    if res['time_interp'] is not None:
        print_kv("Time (interp)", res['time_interp'], fmt="{:.3f}s")
    if res['time_root'] is not None:
        print_kv("Time (root-find)", res['time_root'], fmt="{:.3f}s")

    print_kv("Contains true message", has_true)

    if ranked:
        p0, d0 = ranked[0]
        print_kv("Best distance", d0)
        print_kv("deg(P_best)", p0.degree())
        # brief rationale (no change to logic)
        if has_true:
            print("  Note: t ≤ τ_max here, GS succeeds in this regime.")
        else:
            print("  Note: t > τ_max here; beyond the guaranteed radius.")


print()
print_rule("=")
print("Summary: Algorithm-3 (multiplicities) vs Algorithm-2 (Sudan)")
print_rule("=")

print_algo_block(
    title="Algorithm-3 (Guruswami–Sudan with multiplicities)",
    params={"s": s3, "ell": ell3},
    res=res3, ranked=ranked3, has_true=has_true3, tau=tau3
)

print_algo_block(
    title="Algorithm-2 (Sudan, s=1)",
    params={"s": s2, "ell": ell2},
    res=res2, ranked=ranked2, has_true=has_true2, tau=tau2
)

if not has_true2:
    print()
    if t > tau2:
        print(f"Alg-2 note: t = {t} exceeds τ_max(s=1) = {tau2}, "
              "so the correct codeword need not appear in the list.")
    else:
        print("Alg-2 note: within radius but true message missing (unexpected for this setup).")

Johnson radius τ_J ≈ 27.50
Alg-3 using (s,ℓ)=(2,3) ⇒ τ_max=24
Injecting t = 24 errors (requested 25).
Alg-2 using (s,ℓ)=(1,1) ⇒ τ_max=21

Summary: Algorithm-3 (multiplicities) vs Algorithm-2 (Sudan)

Algorithm-3 (Guruswami–Sudan with multiplicities)
Parameters            : s=2, ℓ=3
GS radius τ_max       : 24
Interpolation degree cap D: 77
# candidates          : 1
Time (interp)         : 0.914s
Time (root-find)      : 0.064s
Contains true message : True
Best distance         : 24
deg(P_best)           : 19
  Note: t ≤ τ_max here, GS succeeds in this regime.

Algorithm-2 (Sudan, s=1)
Parameters            : s=1, ℓ=1
GS radius τ_max       : 21
Interpolation degree cap D: 41
# candidates          : 0
Time (interp)         : 0.121s
Time (root-find)      : 0.011s
Contains true message : False

Alg-2 note: t = 24 exceeds τ_max(s=1) = 21, so the correct codeword need not appear in the list.
